In [3]:
from tabnanny import process_tokens

import pandas as pd
import numpy as np


from db import get_connected

conn = get_connected()

prices = pd.read_sql("SELECT * FROM prices ORDER BY ticker,date", conn)
news= pd.read_sql("SELECT * FROM news ORDER BY ticker, published", conn)
conn.close()

C:\Users\amito\AppData\Local\Temp\ipykernel_10152\1880426003.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  prices = pd.read_sql("SELECT * FROM prices ORDER BY ticker,date", conn)
C:\Users\amito\AppData\Local\Temp\ipykernel_10152\1880426003.py:12: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  news= pd.read_sql("SELECT * FROM news ORDER BY ticker, published", conn)


In [4]:
prices.shape


(5940, 9)

In [5]:
news.shape

(1080, 7)

In [6]:
prices["daily_return"] = prices.groupby("ticker")["close"].pct_change()

In [7]:
prices[["ticker", "date", "close", "daily_return"]].head(10)

,ticker,date,close,daily_return
0,AAPL,2024-05-29,188.655777,NaN
1,AAPL,2024-05-30,189.647202,0.005255
2,AAPL,2024-05-31,190.598969,0.005019
3,AAPL,2024-06-03,192.363647,0.009259
4,AAPL,2024-06-04,192.680923,0.001649
5,AAPL,2024-06-05,194.187866,0.007821
6,AAPL,2024-06-06,192.809799,-0.007097
7,AAPL,2024-06-07,195.199097,0.012392
8,AAPL,2024-06-10,191.461456,-0.019148
9,AAPL,2024-06-11,205.370972,0.072649


In [8]:
prices["ma_7"]= prices.groupby("ticker")["close"].transform(lambda x: x.rolling(7).mean())

In [9]:
prices["ma_21"]= prices.groupby("ticker")["close"].transform(lambda x: x.rolling(21).mean())

In [10]:
prices["ma_ratio"] = prices["ma_7"] / prices["ma_21"]

In [11]:
prices[["ticker", "date", "close", "ma_7", "ma_21", "ma_ratio"]].head(25)

,ticker,date,close,ma_7,ma_21,ma_ratio
0,AAPL,2024-05-29,188.655777,NaN,NaN,NaN
1,AAPL,2024-05-30,189.647202,NaN,NaN,NaN
2,AAPL,2024-05-31,190.598969,NaN,NaN,NaN
3,AAPL,2024-06-03,192.363647,NaN,NaN,NaN
4,AAPL,2024-06-04,192.680923,NaN,NaN,NaN
5,AAPL,2024-06-05,194.187866,NaN,NaN,NaN
6,AAPL,2024-06-06,192.809799,191.563455,NaN,NaN
7,AAPL,2024-06-07,195.199097,192.498215,NaN,NaN
8,AAPL,2024-06-10,191.461456,192.757394,NaN,NaN
9,AAPL,2024-06-11,205.370972,194.867680,NaN,NaN


In [12]:
prices['daily_return'].head()

0         NaN
1    0.005255
2    0.005019
3    0.009259
4    0.001649
Name: daily_return, dtype: float64

In [13]:
prices['volume'].head()

0    53068000.0
1    49889100.0
2    75158300.0
3    50080500.0
4    47471400.0
Name: volume, dtype: float64

In [14]:
prices["volatility_7"] = prices.groupby("ticker")["daily_return"].transform(lambda x: x.rolling(7).std())

In [15]:
prices["volume_change"] = prices.groupby("ticker")["volume"].pct_change()

In [16]:
def calculate_rsi(series, window=14):
    delta = series.diff()
    gain = delta.where(delta > 0, 0)
    loss = -delta.where(delta < 0, 0)
    avg_gain = gain.rolling(window).mean()
    avg_loss = loss.rolling(window).mean()
    rs = avg_gain / avg_loss
    rsi = 100 - (100 / (1 + rs))
    return rsi

In [17]:
prices["rsi"] = prices.groupby("ticker")["close"].transform(calculate_rsi)

In [18]:
prices[["ticker", "date", "close", "rsi"]].head(30)

,ticker,date,close,rsi
0,AAPL,2024-05-29,188.655777,NaN
1,AAPL,2024-05-30,189.647202,NaN
2,AAPL,2024-05-31,190.598969,NaN
3,AAPL,2024-06-03,192.363647,NaN
4,AAPL,2024-06-04,192.680923,NaN
5,AAPL,2024-06-05,194.187866,NaN
6,AAPL,2024-06-06,192.809799,NaN
7,AAPL,2024-06-07,195.199097,NaN
8,AAPL,2024-06-10,191.461456,NaN
9,AAPL,2024-06-11,205.370972,NaN


In [19]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline

sentiment_model = pipeline("sentiment-analysis",
                           model="ProsusAI/finbert",
                           tokenizer="ProsusAI/finbert")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [20]:
test_headlines = [
    "Apple beats earnings expectations, stock surges",
    "Tesla faces massive recall affecting 500,000 vehicles",
    "Microsoft reports steady quarterly revenue"
]

for headline in test_headlines:
    result = sentiment_model(headline)[0]
    print(f"{headline}")
    print(f"  → {result['label']}: {result['score']:.3f}\n")

Apple beats earnings expectations, stock surges
  → positive: 0.825

Tesla faces massive recall affecting 500,000 vehicles
  → negative: 0.953

Microsoft reports steady quarterly revenue
  → positive: 0.904



In [21]:
def get_sentiment_score(headline):
    try:
        result = sentiment_model(headline[:512])[0]  # FinBERT max 512 chars
        if result["label"] == "positive":
            return result["score"]
        elif result["label"] == "negative":
            return -result["score"]
        else:
            return 0.0
    except:
        return 0.0

In [22]:
print("Scoring headlines... this may take a minute")
news["sentiment"] = news["title"].apply(get_sentiment_score)

Scoring headlines... this may take a minute


In [23]:
news["date"] = news["published"].dt.date
daily_sentiment = news.groupby(["ticker", "date"])["sentiment"].mean().reset_index()
daily_sentiment.columns = ["ticker", "date", "sentiment_score"]

In [24]:
daily_sentiment.head()

,ticker,date,sentiment_score
0,AAPL,2026-05-27,0.077354
1,AAPL,2026-05-28,-0.054492
2,AAPL,2026-05-29,0.215910
3,BNB-USD,2026-04-30,0.000000
4,BNB-USD,2026-05-01,0.000000


In [25]:
prices["date"] = pd.to_datetime(prices["date"]).dt.date
features = prices.merge(daily_sentiment, on=["ticker", "date"], how="left")
features["sentiment_score"] = features["sentiment_score"].fillna(0)

In [26]:
features[["ticker", "date", "close", "daily_return", "ma_7", "ma_21", "ma_ratio", "rsi", "volatility_7", "volume_change", "sentiment_score"]].tail(10)

,ticker,date,close,daily_return,ma_7,ma_21,ma_ratio,rsi,volatility_7,volume_change,sentiment_score
5930,TSLA,2026-05-15,422.239990,-0.047507,432.771428,401.286668,1.078460,66.661555,0.035022,0.143656,0.000000
5931,TSLA,2026-05-18,409.989990,-0.029012,432.514282,401.732858,1.076622,62.101888,0.035739,-0.004071,0.000000
5932,TSLA,2026-05-19,404.109985,-0.014342,429.051422,402.285714,1.066534,60.946785,0.031140,-0.113839,0.000000
5933,TSLA,2026-05-20,417.260010,0.032541,425.088566,403.754286,1.052840,62.091903,0.029611,-0.025933,0.000000
5934,TSLA,2026-05-21,417.850006,0.001414,422.859994,405.199047,1.043586,59.741944,0.028760,-0.058678,0.000000
5935,TSLA,2026-05-22,426.010010,0.019529,420.108568,407.689048,1.030463,61.535812,0.027436,0.081333,0.000000
5936,TSLA,2026-05-26,433.589996,0.017793,418.721427,410.417143,1.020234,64.775463,0.028892,-0.009669,-0.432498
5937,TSLA,2026-05-27,440.359985,0.015614,421.309997,413.354761,1.019246,64.155045,0.021522,-0.021954,-0.064542
5938,TSLA,2026-05-28,442.100006,0.003951,425.897143,416.501428,1.022559,61.165550,0.015209,-0.276941,0.052558
5939,TSLA,2026-05-29,438.359985,-0.008460,430.790000,419.623333,1.026611,54.072077,0.013664,-0.055073,-0.597108


In [27]:
features["target"] = features.groupby("ticker")["close"].shift(-1) > features["close"]
features["target"] = features["target"].astype(int)

In [31]:
### How `shift(-1)` creates the target
"""
| date     | close  | shift(-1) (tomorrow's close) | tomorrow > today? | target |
|----------|--------|------------------------------|-------------------|--------|
| May 27   | 440.36 | 442.10                       | 442.10 > 440.36 ✓ | 1 (up) |
| May 28   | 442.10 | 438.36                       | 438.36 > 442.10 ✗ | 0 (down) |
| May 29   | 438.36 | NaN                          | no tomorrow yet   | dropped |

"""

"\n| date     | close  | shift(-1) (tomorrow's close) | tomorrow > today? | target |\n|----------|--------|------------------------------|-------------------|--------|\n| May 27   | 440.36 | 442.10                       | 442.10 > 440.36 ✓ | 1 (up) |\n| May 28   | 442.10 | 438.36                       | 438.36 > 442.10 ✗ | 0 (down) |\n| May 29   | 438.36 | NaN                          | no tomorrow yet   | dropped |\n\n"

In [28]:
feature_cols = ["daily_return", "ma_7", "ma_21", "ma_ratio", "rsi", "volatility_7", "volume_change", "sentiment_score"]
features_clean = features.dropna(subset=feature_cols + ["target"])
features_clean.head()

,id,ticker,date,open,high,low,close,volume,created_at,daily_return,ma_7,ma_21,ma_ratio,volatility_7,volume_change,rsi,sentiment_score,target
20,21,AAPL,2024-06-27,212.846236,213.887222,210.526336,212.261307,49772700.0,2026-05-29 23:16:11.616691,0.003986,209.049120,202.163531,1.034060,0.013660,-0.248296,70.012239,0.0,0
21,22,AAPL,2024-06-28,213.916953,214.214380,208.493929,208.811172,82542700.0,2026-05-29 23:16:11.616691,-0.016254,208.529338,203.123311,1.026615,0.014387,0.658393,63.705308,0.0,1
22,23,AAPL,2024-07-01,210.268521,215.641972,210.099983,214.888504,60402900.0,2026-05-29 23:16:11.616691,0.029104,209.530664,204.325278,1.025476,0.015815,-0.268222,72.526199,0.0,1
23,24,AAPL,2024-07-02,214.293699,218.487382,213.252728,218.378326,58046200.0,2026-05-29 23:16:11.616691,0.016240,211.340700,205.648105,1.027681,0.014689,-0.039016,65.641410,0.0,1
24,25,AAPL,2024-07-03,218.110647,219.647339,217.148976,219.647339,37369800.0,2026-05-29 23:16:11.616691,0.005811,213.239966,206.947328,1.030407,0.014555,-0.356206,61.367310,0.0,1
